In [5]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd
from ocelescope import OCEL

ocel = OCEL.read_ocel(Path("./order-management.sqlite"))

In [14]:
from typing import cast

from ocelescope import OCEL

from ocel_graph.inputs.ocelGraph import Event, OCELGraphInput
from ocel_graph.resources.ocelGraph import EventNode, ObjectNode, OCELGraph, Relation


# Generic function to count neighbours (events or objects)
def group_relation_entity(
    df: pd.DataFrame,
    entity_ids: list[str],
    id_column: str,
    type_column: str,
    target_id_column: str,
):
    """
    Count how many 'target' entities are linked to each entity (event or object).

    Returns:
        DataFrame with columns: id, type, count
    """
    return (
        df[df[id_column].isin(entity_ids)]
        .groupby([id_column, type_column])[target_id_column]
        .size()
        .reset_index(name="count")
        .rename(columns={id_column: "id", type_column: "type"})
    )


def mine_ocel_graph(ocel: OCEL, input: OCELGraphInput):
    graph = OCELGraph()
    root_id = input.root.root

    events_to_visit = []
    objects_to_visit = []

    if isinstance(input.root, Event):
        root = ocel.events[ocel.events[ocel.ocel.event_id_column] == root_id].iloc[0]
        events_to_visit.append(EventNode(id=root_id, activity_type=root[ocel.ocel.event_activity]))
    else:
        root = ocel.objects[ocel.objects[ocel.ocel.object_id_column] == root_id].iloc[0]
        objects_to_visit.append(ObjectNode(id=root_id, object_type=root[ocel.ocel.object_type_column]))

    for _ in range(input.depth):
        # Get current frontier IDs
        event_ids_to_visit = [event.id for event in events_to_visit]
        object_ids_to_visit = [obj.id for obj in objects_to_visit]

        # Get event-object relations using XOR and not already in the graph
        relations: pd.DataFrame = cast(
            pd.DataFrame,
            ocel.relations[
                (
                    (ocel.relations[ocel.ocel.event_id_column].isin(event_ids_to_visit))
                    ^ (ocel.relations[ocel.ocel.object_id_column].isin(object_ids_to_visit))
                )
                & ~(ocel.relations[ocel.ocel.event_id_column].isin(graph.event_ids))
                & ~(ocel.relations[ocel.ocel.object_id_column].isin(graph.object_ids))
            ],
        )

        # Count object neighbors per event
        events = group_relation_entity(
            df=relations,
            entity_ids=event_ids_to_visit,
            id_column=ocel.ocel.event_id_column,
            type_column=ocel.ocel.event_activity,
            target_id_column=ocel.ocel.object_id_column,
        )

        # Count event neighbors per object
        e2o_objects = group_relation_entity(
            df=relations,
            entity_ids=object_ids_to_visit,
            id_column=ocel.ocel.object_id_column,
            type_column=ocel.ocel.object_type_column,
            target_id_column=ocel.ocel.event_id_column,
        )

        # Get object-object (o2o) relations using XOR
        o2o = cast(
            pd.DataFrame,
            ocel.o2o[
                (
                    (ocel.o2o["ocel:oid_1"].isin(object_ids_to_visit))
                    ^ (ocel.o2o["ocel:oid_2"].isin(object_ids_to_visit))
                )
                & ~(ocel.o2o["ocel:oid_1"].isin(graph.object_ids))
                & ~(ocel.o2o["ocel:oid_2"].isin(graph.object_ids))
            ],
        )

        # Normalize and mirror o2o (treat as undirected)
        o2o = o2o.rename(
            columns={"ocel:oid_1": "id", "ocel:type_1": "type", "ocel:oid_2": "target_id", "ocel:type_2": "target_type"}
        )

        mirrored = o2o.rename(
            columns={"target_id": "id", "target_type": "type", "id": "target_id", "type": "target_type"}
        )

        mirrored_o2o = pd.concat([o2o, mirrored], ignore_index=True).rename(columns={"ocel:qualifier": "qualifier"})

        # Count o2o neighbours for each object
        o2o_objects = group_relation_entity(
            df=mirrored_o2o,
            entity_ids=object_ids_to_visit,
            id_column="id",
            type_column="type",
            target_id_column="target_id",
        )

        # Combine object neighbour counts
        objects = (
            pd.concat([o2o_objects, e2o_objects], ignore_index=True)
            .groupby(["id", "type"], as_index=False)["count"]
            .sum()
        )

        # Update graph with this layer
        graph.objects.update(objects_to_visit)
        graph.events.update(events_to_visit)

        # Prepare for next layer
        events_to_visit = []
        objects_to_visit = []

        object_id_with_neighbours = [
            row["id"] for _, row in objects.iterrows() if row["count"] <= input.max_neighbours or root_id == row[id]
        ]
        event_id_with_neighbours = [
            row["id"] for _, row in events.iterrows() if row["count"] <= input.max_neighbours or root_id == row[id]
        ]

        # Add o2o relations to graph and queue new objects
        for _, row in (
            cast(pd.DataFrame, mirrored_o2o[mirrored_o2o["id"].isin(object_id_with_neighbours)])
            .drop_duplicates(subset=["target_id"], keep="first")
            .iterrows()
        ):
            graph.o2oRelations.add(
                Relation(source=str(row["id"]), target=str(row["target_id"]), qualifier=str(row["qualifier"]))
            )
            objects_to_visit.append(ObjectNode(id=str(row["target_id"]), object_type=str(row["target_type"])))

        # Add e2o relations (object → event)
        for _, row in (
            cast(pd.DataFrame, relations[relations["ocel:oid"].isin(object_id_with_neighbours)])
            .drop_duplicates(subset=["ocel:eid"], keep="first")
            .iterrows()
        ):
            graph.e2oRelations.add(
                Relation(
                    source=str(row["ocel:eid"]),
                    target=str(row["ocel:oid"]),
                    qualifier=str(row["ocel:qualifier"]),
                )
            )
            events_to_visit.append(EventNode(id=str(row["ocel:eid"]), activity_type=str(row["ocel:activity"])))

        # Add e2o relations (event → object)
        for _, row in (
            cast(
                pd.DataFrame,
                relations[
                    relations["ocel:eid"].isin(event_id_with_neighbours)
                    & ~relations["ocel:oid"].isin([obj.id for obj in objects_to_visit])
                ],
            )
            .drop_duplicates(subset=["ocel:oid"], keep="first")
            .iterrows()
        ):
            graph.e2oRelations.add(
                Relation(
                    source=str(row["ocel:eid"]),
                    target=str(row["ocel:oid"]),
                    qualifier=str(row["ocel:qualifier"]),
                )
            )
            objects_to_visit.append(ObjectNode(id=str(row["ocel:oid"]), object_type=str(row["ocel:type"])))
    return graph

In [15]:
mine_ocel_graph(ocel, OCELGraphInput(depth=3, max_neighbours=10, root=Event(root="place_o-990001")))

OCELGraph(events={EventNode(id='create_p-660001', activity_type='create package'), EventNode(id='send_p-660001', activity_type='send package'), EventNode(id='pick_i-880002', activity_type='pick item'), EventNode(id='confirm_o-990001', activity_type='confirm order'), EventNode(id='deliver_p-660001', activity_type='package delivered'), EventNode(id='pick_i-880001', activity_type='pick item'), EventNode(id='pay_o-990001', activity_type='pay order'), EventNode(id='pick_i-880003', activity_type='pick item'), EventNode(id='place_o-990001', activity_type='place order')}, objects={ObjectNode(id='iPhone 11 Pro', object_type='products'), ObjectNode(id='i-880003', object_type='items'), ObjectNode(id='AlpenTech Innovations AG', object_type='customers'), ObjectNode(id='p-660001', object_type='packages'), ObjectNode(id='i-880001', object_type='items'), ObjectNode(id='i-880002', object_type='items'), ObjectNode(id='Echo', object_type='products'), ObjectNode(id='o-990001', object_type='orders')}, e2oR